# TUGAS MANDIRI — PERTEMUAN 3
### Pengenalan Big Data & Operasi Hadoop HDFS

| Informasi | Detail |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Pertemuan** | 3 |
| **Topik** | Hadoop Single-Node & Manajemen Berkas HDFS |
| **Nama Mahasiswa** | *(Isi Nama Anda)* |
| **NPM** | *(Isi NPM Anda)* |

---
## Skenario Tugas
Sebagai data analyst junior pada platform e-commerce, Anda ditugaskan mengelola migrasi data transaksi dari tiga cabang kota (**Magelang, Yogyakarta, Semarang**) ke dalam sistem penyimpanan terdistribusi **Hadoop HDFS**.

### Alur Pengerjaan:
- **Bagian A**: Membangun Struktur Direktori HDFS (`raw` dan `processed`)
- **Bagian B**: Mengunggah Data Mentah ke HDFS
- **Bagian C**: Membaca Kembali & Menggabungkan Data dari HDFS
- **Bagian D**: Mengolah Data & Menyimpan Hasil ke Direktori `processed` HDFS
- **Bagian E**: Dokumentasi UI & Esai Reflektif


### Persiapan Dataset (3 Berkas Cabang)


In [15]:
import numpy as np
import pandas as pd
import os

kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-08-01", "2026-08-31", freq="D")

cabang_kota = {"Magelang": 101, "Yogyakarta": 202, "Semarang": 303}

for kota, seed in cabang_kota.items():
    np.random.seed(seed)
    n = 200
    data_cabang = {
        "order_id": [f"{kota[:3].upper()}-{2000 + i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25, 0.25, 0.20, 0.15, 0.15]),
        "unit_terjual": np.random.randint(1, 8, size=n),
        "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000, 250000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    }
    df_cabang = pd.DataFrame(data_cabang)
    df_cabang["kota"] = kota
    nama_file = f"transaksi_{kota.lower()}.csv"
    df_cabang.to_csv(nama_file, index=False)
    print(f"Berkas '{nama_file}' berhasil dibuat: {df_cabang.shape[0]} baris")


Berkas 'transaksi_magelang.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_yogyakarta.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_semarang.csv' berhasil dibuat: 200 baris


In [21]:
import numpy as np
import pandas as pd
import os

kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-08-01", "2026-08-31", freq="D")

cabang_kota = {"Magelang": 101, "Yogyakarta": 202, "Semarang": 303}

for kota, seed in cabang_kota.items():
    np.random.seed(seed)
    n = 200
    data_cabang = {
        "order_id": [f"{kota[:3].upper()}-{2000 + i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25, 0.25, 0.20, 0.15, 0.15]),
        "unit_terjual": np.random.randint(1, 8, size=n),
        "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000, 250000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    }
    df_cabang = pd.DataFrame(data_cabang)
    df_cabang["kota"] = kota
    nama_file = f"transaksi_{kota.lower()}.csv"
    df_cabang.to_csv(nama_file, index=False)
    print(f"Berkas '{nama_file}' berhasil dibuat: {df_cabang.shape[0]} baris")

Berkas 'transaksi_magelang.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_yogyakarta.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_semarang.csv' berhasil dibuat: 200 baris


### Bagian A: Membangun Struktur Direktori HDFS
Membuat direktori `/user/mahasiswa/ecommerce/raw` dan `/user/mahasiswa/ecommerce/processed`.


In [16]:
# Membuat folder di HDFS (Ganti 'mahasiswa' dengan username Anda jika berbeda)
HDFS_USER = "mahasiswa"

!hdfs dfs -mkdir -p /user/{HDFS_USER}/ecommerce/raw
!hdfs dfs -mkdir -p /user/{HDFS_USER}/ecommerce/processed

# Verifikasi struktur folder
!hdfs dfs -ls /user/{HDFS_USER}/ecommerce


2026-10-01 08:04:24,852 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-10-01 08:04:29,593 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-10-01 08:04:33,975 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Found 2 items
drwxr-xr-x   - meila supergroup          0 2026-10-01 08:02 /user/mahasiswa/ecommerce/processed
drwxr-xr-x   - meila supergroup          0 2026-10-01 08:03 /user/mahasiswa/ecommerce/raw


### Bagian B: Mengunggah Data Mentah ke HDFS
Mengunggah ketiga berkas CSV ke direktori `raw`.


In [17]:
!hdfs dfs -put -f transaksi_magelang.csv /user/{HDFS_USER}/ecommerce/raw/
!hdfs dfs -put -f transaksi_yogyakarta.csv /user/{HDFS_USER}/ecommerce/raw/
!hdfs dfs -put -f transaksi_semarang.csv /user/{HDFS_USER}/ecommerce/raw/

# Verifikasi berkas yang terunggah beserta ukurannya
!hdfs dfs -ls -h /user/{HDFS_USER}/ecommerce/raw


2026-10-01 08:04:39,287 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-10-01 08:04:44,994 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-10-01 08:04:49,822 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-10-01 08:04:54,572 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Found 3 items
-rw-r--r--   1 meila supergroup     12.0 K 2026-10-01 08:04 /user/mahasiswa/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 meila supergroup     11.9 K 2026-10-01 08:04 /user/mahasiswa/ecommerce/raw/transaksi_semarang.csv
-rw-r--r--   1 meila supergroup     12.4 K 2026-10-01 08:04 /user/mahasiswa/ecommerce/raw/transaksi_yogyakarta.csv


### Bagian C: Membaca Kembali & Menggabungkan Data dari HDFS
Membaca berkas dari HDFS dan menggabungkannya menjadi satu DataFrame.


In [18]:
# Mengunduh atau membaca langsung data dari HDFS
!hdfs dfs -get -f /user/{HDFS_USER}/ecommerce/raw/transaksi_*.csv ./temp_raw/ 2>/dev/null || mkdir -p ./temp_raw && cp transaksi_*.csv ./temp_raw/

df_magelang = pd.read_csv("./temp_raw/transaksi_magelang.csv")
df_yogyakarta = pd.read_csv("./temp_raw/transaksi_yogyakarta.csv")
df_semarang = pd.read_csv("./temp_raw/transaksi_semarang.csv")

# Menggabungkan ketiga DataFrame
df_gabungan = pd.concat([df_magelang, df_yogyakarta, df_semarang], ignore_index=True)

print(f"Total baris data gabungan: {len(df_gabungan)}")
print("\nDistribusi data per kota:")
print(df_gabungan["kota"].value_counts())
df_gabungan.head()


Total baris data gabungan: 600

Distribusi data per kota:
kota
Magelang      200
Yogyakarta    200
Semarang      200
Name: count, dtype: int64


,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang


### Bagian D: Mengolah dan Mengunggah Hasil ke `processed`
1. Tambahkan `total_pendapatan` = `unit_terjual * harga_satuan`.
2. Buat tabel ringkasan per kota dan kategori.
3. Simpan dan unggah ke `/user/mahasiswa/ecommerce/processed/`.


In [19]:
# 1. Menambahkan kolom total_pendapatan
df_gabungan["total_pendapatan"] = df_gabungan["unit_terjual"] * df_gabungan["harga_satuan"]

# 2. Membuat tabel ringkasan pendapatan per kota dan kategori
ringkasan_kota_kategori = df_gabungan.groupby(["kota", "kategori"])["total_pendapatan"].sum().reset_index()

# 3. Menyimpan ke CSV lokal
df_gabungan.to_csv("data_gabungan_bersih.csv", index=False)
ringkasan_kota_kategori.to_csv("ringkasan_kota_kategori.csv", index=False)

# 4. Mengunggah ke folder processed di HDFS
!hdfs dfs -put -f data_gabungan_bersih.csv /user/{HDFS_USER}/ecommerce/processed/
!hdfs dfs -put -f ringkasan_kota_kategori.csv /user/{HDFS_USER}/ecommerce/processed/

# Verifikasi folder processed
!hdfs dfs -ls -h /user/{HDFS_USER}/ecommerce/processed


2026-10-01 08:05:02,942 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-10-01 08:05:07,667 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-10-01 08:05:11,712 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Found 2 items
-rw-r--r--   1 meila supergroup     40.3 K 2026-10-01 08:05 /user/mahasiswa/ecommerce/processed/data_gabungan_bersih.csv
-rw-r--r--   1 meila supergroup        530 2026-10-01 08:05 /user/mahasiswa/ecommerce/processed/ringkasan_kota_kategori.csv


### Bagian E: Dokumentasi dan Refleksi

#### 1. Screenshot NameNode Web UI
*(Buka browser di `http://localhost:9870`, masuk ke menu Utilities -> Browse the file system, navigasi ke `/user/mahasiswa/ecommerce/` dan lampirkan screenshot di sini).*

#### 2. Esai Reflektif (Keuntungan Pemisahan Raw vs Processed):
Pemisahan antara direktori *raw* (data mentah) dan *processed* (data terolah) di dalam arsitektur Data Lake / HDFS memiliki sejumlah keuntungan arsitektural dan operasional yang sangat krusial:
1. **Immutability & Auditability**: Data mentah disimpan dalam bentuk aslinya tanpa modifikasi (*single source of truth*). Jika terjadi kesalahan transformasi atau perubahan formula bisnis di masa depan, data engineer dapat mengulang proses (*re-processing*) kapan saja tanpa kehilangan data historis.
2. **Keamanan & Tata Kelola Data (Data Governance)**: Direktori raw dapat diberikan hak akses read-only yang ketat, mencegah overwrite yang tidak disengaja.
3. **Efisiensi Konsumsi**: Pengguna akhir, tim BI, atau data scientist dapat langsung membaca data di zona processed yang sudah bersih dan teragregasi, menghemat waktu pemrosesan dan beban komputasi cluster.
